# core

> Workflow-level route handlers for status, reset, and source retrieval

In [ ]:
#| default_exp routes.core

In [ ]:
#| export
from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

## Core Route Handlers

These handlers manage workflow status and lifecycle.

In [ ]:
#| export
def _handle_current_status(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess  # FastHTML session object
):  # Appropriate UI component based on current state
    """Return current workflow status - determines what to show."""
    session_id = get_session_id(sess)
    
    # Check for in-progress workflow (via server-side state store)
    workflow_state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
    current_step = workflow.state_store.get_current_step(workflow.config.workflow_id, session_id)
    
    # Restore external paths from state to SourceService (fixes persistence across restarts)
    step_states = workflow_state.get("step_states", {})
    selection_state = step_states.get("selection", {})
    external_db_paths = selection_state.get("external_db_paths", [])
    if external_db_paths:
        workflow.source_service.set_external_paths(external_db_paths)
    
    # If there's existing state, resume the workflow
    if current_step or workflow_state:
        return workflow._stepflow_router.start(request, sess)
    
    # Start fresh
    return workflow._stepflow_router.start(request, sess)

In [ ]:
#| export
def _handle_reset(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess  # FastHTML session object
):  # StepFlow start view
    """Reset workflow and return to start."""
    session_id = get_session_id(sess)
    workflow.state_store.clear_state(workflow.config.workflow_id, session_id)
    return workflow._stepflow_router.start(request, sess)

In [ ]:
#| export
def _handle_get_sources(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    plugin_name: str = None,  # Optional plugin name filter
    limit: int = 50  # Maximum number of results
):  # JSON response with transcription sources
    """Get available transcription sources."""
    transcriptions = workflow.source_service.query_transcriptions(
        plugin_name=plugin_name,
        limit=limit
    )
    return {"transcriptions": transcriptions}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()